<div align="center">

# Universidad Autónoma de Chihuahua
## Facultad de Ingeniería
### Maestría en Ingeniería en Computación

**Materia:** Machine Learning (MIC1)

**Actividad:** Proyecto Final — Notebook 4: Resultados y análisis

**Alumno:** Jonahtan Gabriel González Cardoza
**Matrícula:** 345491

**Catedrática:** Dra. Olanda Prieto Ordaz

**Fecha de elaboración:** 21 de noviembre de 2026

</div>

**Proyecto:** predicción de la situación final de un estudiante (*Dropout*, *Enrolled*, *Graduate*) con el dataset
*Predict Students' Dropout and Academic Success* (UCI 697). Este proyecto reúne, en cinco notebooks reproducibles, el trabajo
de las tres prácticas anteriores (regresión logística del Proyecto F_1, SVM y métodos de ensamble).

**Objetivo de este notebook.** Evaluar los modelos seleccionados en el notebook 03: predicciones *out-of-fold* en Train, validación cruzada repetida con prueba t corregida, **evaluación única en Test** con intervalos de confianza y McNemar, generalización, interpretabilidad y comparación con la literatura.

# III. Desarrollo (continuación)

In [ ]:
from config import *

X_train, X_test, y_train, y_test = cargar_particion()
numeric_cols, nominal_cols, binary_cols = columnas()

# Modelos, predicciones OOF y búsquedas generados por el notebook 03
reg = Registro.cargar(X_train, y_train)
resultados_cv, oof, modelos_ajustados = reg.resultados_cv, reg.oof, reg.modelos_ajustados
searches, tiempos_busqueda = reg.searches, reg.tiempos_busqueda
tabla_resumen, tabla_por_clase = reg.tabla_resumen, reg.tabla_por_clase
UMBRAL_LR = reg.extra["umbral_lr"]               # regla de decisión de la LR, elegida con OOF de Train (notebook 03)
modelos_defecto = [n for n in modelos_ajustados if n.endswith("(por defecto)")]
print(f"{len(modelos_ajustados)} modelos cargados | Train {X_train.shape} | Test {X_test.shape} | regla de la LR: P(Dropout) ≥ {UMBRAL_LR}")

## Fase 7. Evaluación sobre Train mediante predicciones Out-of-Fold

### 22. Los modelos finales contra la regresión logística (OOF)
Cada estudiante de Train se predice con un modelo que **no** lo vio al entrenar. Todos los modelos usan las mismas predicciones OOF (los mismos 5 folds) y su regla por defecto (la clase con mayor puntaje o más votada). Las métricas se calculan sobre las 3,539 predicciones juntas (por eso difieren ligeramente del promedio de los 5 folds).

In [ ]:
# Criterio declarado de antemano: entre los ensambles optimizados, el de mayor F1 macro en CV (5 folds)
ensambles_opt = [n for n in resultados_cv if n.endswith("optimizado") and not n.startswith("SVM")]
NOMBRE_FINAL = max(ensambles_opt, key=lambda n: resultados_cv[n]["F1_mean"])
NOMBRE_SVM = "SVM RBF optimizado"
print("Ensambles optimizados por F1 macro en CV (5 folds):")
for n in sorted(ensambles_opt, key=lambda n: -resultados_cv[n]["F1_mean"]):
    print(f"  {n:<36s} {resultados_cv[n]['F1_mean']:.4f} ± {resultados_cv[n]['F1_std']:.4f}")
print(f"\nEnsamble final (criterio: F1 macro en CV): {NOMBRE_FINAL}")

nombres_oof = [NOMBRE_LR, NOMBRE_SVM] + sorted(ensambles_opt, key=lambda n: -resultados_cv[n]["F1_mean"])
tabla_oof = pd.DataFrame({n: metricas(y_train, *oof[n]) for n in nombres_oof}).T
print("\nMétricas sobre las predicciones out-of-fold de Train (3,539 estudiantes; regla por defecto de cada modelo):")
display(tabla_oof.round(4))
for metrica in ["Recall", "Precision", "F1"]:
    print(f"{metrica} por clase (OOF):")
    display(tabla_por_clase(metrica, nombres_oof).round(4))

mostrar = [NOMBRE_LR, NOMBRE_SVM, "Random Forest optimizado", "HistGradientBoosting optimizado", NOMBRE_FINAL, "EBM optimizado"]
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
for ax, nombre in zip(axes.ravel(), mostrar):
    ConfusionMatrixDisplay(confusion_matrix(y_train, oof[nombre][0]), display_labels=CLASES).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"OOF — {nombre}", fontsize=10.5)
    ax.set_xlabel("Clase predicha")
    ax.set_ylabel("Clase real")
    ax.grid(False)
plt.tight_layout()
guardar_fig("04_oof_matrices_confusion")

pred_final, score_final = oof[NOMBRE_FINAL]
pred_lr, score_lr = oof[NOMBRE_LR]
print(f"¿En qué coinciden? {NOMBRE_FINAL} (filas) contra la regresión logística (columnas), OOF:")
display(pd.crosstab(pd.Categorical.from_codes(pred_final, CLASES), pd.Categorical.from_codes(pred_lr, CLASES),
                    rownames=["Ensamble final"], colnames=["LR"]))
print(f"Coinciden en {np.mean(pred_final == pred_lr):.1%} de los estudiantes")

modelos_roc_oof = [NOMBRE_LR, "Random Forest optimizado", "Gradient Boosting optimizado", "HistGradientBoosting optimizado",
                   "EBM optimizado", NOMBRE_FINAL, "Stacking optimizado"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (k, clase) in zip(axes, enumerate(CLASES)):
    for n, color in zip(modelos_roc_oof, sns.color_palette("tab10", len(modelos_roc_oof))):
        sc = oof[n][1]
        fpr, tpr, _ = roc_curve(y_train == k, sc[:, k])
        ax.plot(fpr, tpr, lw=1.8, color=color, label=f"{n.replace(' optimizado', '')} (AUC={roc_auc_score(y_train == k, sc[:, k]):.3f})")
    ax.plot([0, 1], [0, 1], color=COLOR_GRIS, lw=1)
    ax.set_title(f"ROC OOF — {clase} contra el resto")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right", fontsize=7.8)
plt.tight_layout()
guardar_fig("04_oof_roc")
guardar_tabla(tabla_oof, "04_oof_metricas")

**Interpretación.**

- **El orden por métricas OOF coincide con el de la CV** (son las mismas predicciones de los 5 modelos de cada fold, juntadas): *Voting soft* 0.7377 de F1 macro, Accuracy 0.7858, MCC 0.6506 y ROC-AUC 0.9009, contra 0.7171, 0.7584, 0.6172 y 0.8905 de la regresión logística. Con más Accuracy y más MCC, los ensambles ganan la comparación; con F1 macro la ventaja es de 0.02 y con *recall* macro de 0.01.
- **Coinciden en 89.9% de los estudiantes** (3,181 de 3,539). **El 69% de los 358 desacuerdos son estudiantes a los que la regresión logística etiqueta como *Enrolled***: 167 que el ensamble manda a *Graduate* y 80 que manda a *Dropout*. Es el efecto de `class_weight = "balanced"` en la regresión logística: tira de los casos dudosos hacia la clase intermedia. En las otras cuatro celdas hay 111 desacuerdos (30, 30, 36 y 15).
- **Hay un intercambio entre Accuracy y *recall* de *Enrolled*.** Los modelos con más Accuracy reconocen menos *Enrolled* (SVM RBF 43.6%, *Voting soft* 55.6%, contra 61.1% de la regresión logística), y los que más reconocen (EBM 65.0%, AdaBoost 63.3%, Stacking 61.4%) no encabezan la Accuracy. Es el mismo patrón que en el SVM: se elige entre equivocarse menos en la clase mayoritaria o rescatar a la intermedia.
- **Las ganancias de los ensambles están en *Dropout* y *Graduate*, no en *Enrolled*.** El F1 de *Dropout* pasa de 0.783 (regresión logística) a hasta 0.807 en los mejores ensambles, mientras que el de *Enrolled* se queda entre 0.51 y 0.56 en todos (0.527 la regresión logística). *Enrolled* sigue siendo la clase difícil, sin importar qué ensamble se use.

### 23. ¿Las diferencias son reales? (validación cruzada repetida)
Con solo 5 folds, una diferencia de un par de centésimas puede ser ruido. Para compararlos con la regresión logística se repite la validación cruzada estratificada 5 veces con distintas particiones (5 × 5 = 25 evaluaciones **pareadas**: todos los modelos usan exactamente los mismos folds) y se aplica la **prueba t corregida de Nadeau y Bengio (2003)**. La corrección es necesaria porque los folds de entrenamiento se traslapan y la prueba t ordinaria subestimaría la varianza:

$$t = \frac{\bar d}{\sqrt{\left(\frac{1}{k} + \frac{n_\text{test}}{n_\text{train}}\right)\hat\sigma^2_d}}, \qquad k = 25,\; \frac{n_\text{test}}{n_\text{train}} = \frac{1}{4}$$

donde $d$ es la diferencia (modelo − regresión logística) en cada evaluación. Como se hacen **11 comparaciones** contra la misma referencia, además del p-valor se reporta el ajustado por el método de **Holm** (que controla la probabilidad de al menos un falso positivo). Todos los modelos usan su regla por defecto; el *Voting hard* no tiene ROC-AUC. Por costo, se repite sobre los 10 modelos finales y el SVM; el *Stacking* es el más lento (≈10 min).

In [ ]:
# Validación cruzada repetida: 5 repeticiones × 5 folds = 25 evaluaciones pareadas (los mismos folds para todos los modelos)
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=K_REPETICIONES, random_state=RANDOM_STATE)
K_REP = rcv.get_n_splits()
metricas_rep = ["Accuracy", "Recall", "F1", "MCC", "ROC-AUC"]
modelos_rep = [NOMBRE_LR, "SVM RBF optimizado", "Random Forest optimizado", "Bagging optimizado", "Pasting optimizado",
               "AdaBoost optimizado", "Gradient Boosting optimizado", "HistGradientBoosting optimizado",
               "EBM optimizado", "Voting (hard) optimizado", "Voting (soft) optimizado", "Stacking optimizado"]
rep, tiempos_rep = {}, {}
for nombre in modelos_rep:
    t0 = time.perf_counter()
    sc = {m: scoring[m] for m in metricas_rep if not (m == "ROC-AUC" and "hard" in nombre)}   # Voting hard: sin puntaje continuo
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        rep[nombre] = cross_validate(clone(modelos_ajustados[nombre]), X_train, y_train, cv=rcv, n_jobs=N_JOBS, scoring=sc)
    tiempos_rep[nombre] = time.perf_counter() - t0
    print(f"{nombre}: F1 macro {rep[nombre]['test_F1'].mean():.4f} ± {rep[nombre]['test_F1'].std(ddof=1):.4f} "
          f"({tiempos_rep[nombre]:.0f} s)", flush=True)

In [ ]:
def prueba_corregida(d):
    """Prueba t pareada con la corrección de Nadeau y Bengio (2003): n_test / n_train = 1/4 en 5 folds."""
    var_corr = (1 / K_REP + 1 / 4) * d.var(ddof=1)
    t_stat = d.mean() / np.sqrt(var_corr)
    h = stats.t.ppf(0.975, K_REP - 1) * np.sqrt(var_corr)
    return d.mean(), d.mean() - h, d.mean() + h, t_stat, 2 * stats.t.sf(abs(t_stat), df=K_REP - 1)

# (1) Media ± desviación de las 25 evaluaciones
resumen_rep = pd.DataFrame({n: {m: (f"{rep[n][f'test_{m}'].mean():.4f} ± {rep[n][f'test_{m}'].std(ddof=1):.4f}"
                                    if f"test_{m}" in rep[n] else "—") for m in metricas_rep}
                            for n in modelos_rep}).T
resumen_rep["F1 (CV 5 folds, búsqueda)"] = [resultados_cv[n]["F1_mean"] for n in modelos_rep]
resumen_rep["Optimismo de la búsqueda (5 folds − 5×5)"] = [resultados_cv[n]["F1_mean"] - rep[n]["test_F1"].mean()
                                                          for n in modelos_rep]
resumen_rep["_orden"] = [rep[n]["test_F1"].mean() for n in modelos_rep]
print("Validación cruzada repetida (5 × 5 = 25 evaluaciones), media ± desviación:")
display(resumen_rep.sort_values("_orden", ascending=False).drop(columns="_orden").round(4))

# (2) F1 macro: cada modelo contra la regresión logística (regla por defecto), prueba t corregida + Holm
filas = {}
for n in modelos_rep[1:]:
    d = rep[n]["test_F1"] - rep[NOMBRE_LR]["test_F1"]
    media, lo, hi, t_stat, p = prueba_corregida(d)
    filas[n] = {"Diferencia F1 vs LR": media, "IC 95% inferior": lo, "IC 95% superior": hi,
                "Gana (de 25)": int((d > 0).sum()), "t corregida": t_stat, "p-valor": p}
comp_f1 = pd.DataFrame(filas).T
orden = comp_f1["p-valor"].sort_values().index
holm = np.maximum.accumulate([(len(orden) - i) * comp_f1.loc[n, "p-valor"] for i, n in enumerate(orden)])
comp_f1["p Holm"] = pd.Series(np.minimum(holm, 1.0), index=orden)
print(f"F1 macro: cada modelo − {NOMBRE_LR} (25 evaluaciones pareadas; p Holm corrige las {len(orden)} comparaciones)")
display(comp_f1.sort_values("Diferencia F1 vs LR", ascending=False).round(4))

# (3) Las otras métricas contra la regresión logística: diferencia media y p-valor sin corregir
filas = {}
for n in modelos_rep[1:]:
    fila = {}
    for m in ["Accuracy", "Recall", "MCC", "ROC-AUC"]:
        if f"test_{m}" in rep[n]:
            media, _, _, _, p = prueba_corregida(rep[n][f"test_{m}"] - rep[NOMBRE_LR][f"test_{m}"])
            fila[m] = f"{media:+.4f} (p={p:.3f})"
        else:
            fila[m] = "—"
    filas[n] = fila
print("Otras métricas, diferencia contra la LR: media (p-valor sin corregir)")
display(pd.DataFrame(filas).T)

# (4) Distribución de las 25 evaluaciones de F1 macro
orden_plot = sorted(modelos_rep, key=lambda n: rep[n]["test_F1"].mean())
fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot([rep[n]["test_F1"] for n in orden_plot], orientation="horizontal", tick_labels=[n.replace(" optimizado", "") for n in orden_plot])
ax.axvline(rep[NOMBRE_LR]["test_F1"].mean(), color="tab:red", ls="--", lw=1.2, label="media de la LR")
ax.set_xlabel("F1 macro (25 evaluaciones)")
ax.set_title("Validación cruzada repetida 5×5 — F1 macro")
ax.legend()
plt.tight_layout()
guardar_fig("04_cv_repetida_boxplot")
guardar_tabla(resumen_rep.drop(columns="_orden"), "04_cv_repetida_resumen")
guardar_tabla(comp_f1, "04_cv_repetida_vs_lr_f1")

**Interpretación.**

- **Ningún ensamble supera de forma significativa a la regresión logística en F1 macro una vez corregidas las 11 comparaciones.** El mejor, *Voting soft*, tiene +0.0160 (IC 95% [0.002, 0.030]) y gana en 22 de las 25 evaluaciones, con p = 0.024 sin corregir pero **p de Holm = 0.26**. Le siguen *Voting hard* (+0.0152, p = 0.050), Stacking (+0.0127, p = 0.069), HistGradientBoosting (+0.0093) y Gradient Boosting (+0.0090); EBM (+0.0057) y Random Forest (+0.0027) están en el ruido, y Pasting, AdaBoost, SVM RBF y Bagging quedan ligeramente por debajo (−0.004 a −0.010).
- **Las cifras de la CV de 5 folds son optimistas por haber sido elegidas como máximo.** La columna "Optimismo" compara el F1 de la búsqueda (5 folds) con el de la CV repetida: de 0.0018 (Bagging) a 0.0092 (SVM RBF) en los modelos que se optimizaron, 0.0044 en promedio entre los ensambles; en la regresión logística, que no se optimizó aquí, es −0.0011. Es la *maldición del ganador*: de varias configuraciones con rendimiento parecido, la elegida es la que tuvo suerte en esos folds. Cerca de una quinta parte de la ventaja de *Voting soft* en la CV de 5 folds (+0.0205 contra la regresión logística) desaparece en la CV repetida (+0.0160).
- **Con otras métricas, la conclusión cambia, y esto debe leerse con cuidado.** Sin corregir, *Voting soft* es mejor que la regresión logística en Accuracy (+0.0249, p < 0.001), MCC (+0.0290, p = 0.002) y ROC-AUC (+0.0090, p = 0.001), y Stacking en ROC-AUC (+0.0073, p = 0.005). Con la corrección de Holm por las 11 comparaciones, solo la Accuracy de *Voting soft* y del SVM RBF (+0.0231, p = 0.002) seguiría bajo 0.05 (cuenta aproximada, con los p-valores redondeados). En *recall* macro ningún ensamble es distinto de la regresión logística, y el SVM RBF (−0.0240, p = 0.014) y Bagging (−0.0197, p = 0.001) son peores. Los ensambles se parecen más a la regresión logística en lo que pesa a las tres clases por igual, y la superan algo en las métricas que pesan por tamaño de clase.
- **Las diferencias son del tamaño de la desviación entre evaluaciones** (0.012 a 0.018 en la CV repetida), así que con este tamaño de datos (3,539 estudiantes) no se puede ordenar a los ensambles entre sí con la CV.

## Fase 8. Evaluación final sobre Test

### 24. Evaluación en Test de todos los modelos
Concluida la selección, se usa Test una sola vez. Todos los modelos ya estaban seleccionados con CV; **ninguna decisión se toma con esta tabla**. El Pipeline aplica el preprocesamiento aprendido con Train. La regresión logística aparece dos veces: con su regla por defecto y con la regla P(*Dropout*) ≥ 0.40 que se fijó en el Proyecto F_1, que es su resultado oficial.

In [ ]:
# Todos los modelos (ya ajustados con Train y seleccionados con CV) se evalúan una sola vez sobre el mismo Test
preds_test, scores_test, tiempos_pred = {}, {}, {}
for nombre in modelos_ajustados:
    modelo = modelos_ajustados[nombre]
    t0 = time.perf_counter()
    preds_test[nombre] = modelo.predict(X_test)
    scores_test[nombre] = puntaje_continuo(modelo, X_test)
    tiempos_pred[nombre] = time.perf_counter() - t0
NOMBRE_LR_REGLA = f"{NOMBRE_LR} + P(Dropout) ≥ {UMBRAL_LR:.2f}"
preds_test[NOMBRE_LR_REGLA] = predecir_regla(scores_test[NOMBRE_LR], UMBRAL_LR)
scores_test[NOMBRE_LR_REGLA], tiempos_pred[NOMBRE_LR_REGLA] = scores_test[NOMBRE_LR], tiempos_pred[NOMBRE_LR]

grupos_test = {
    "Referencias (Dummy, regresión logística, SVM)": ["Dummy (referencia)", NOMBRE_LR, NOMBRE_LR_REGLA,
                                                     "SVM Lineal optimizado", "SVM Polinomial optimizado", "SVM RBF optimizado"],
    "Modelos base por defecto (referencia adicional)": list(modelos_defecto),
    "Ensambles con hiperparámetros base": ["Árbol de decisión (1 árbol)", "Voting (hard)", "Voting (soft)", "Bagging (árboles)",
                                           "Pasting (árboles)", "Random Forest", "AdaBoost", "Gradient Boosting (balanceado)",
                                           "HistGradientBoosting (balanceado)", "Stacking (LR + SVM + RF → LR)",
                                           "EBM sin interacciones (GAM)", "EBM (10 interacciones)"],
    "Ensambles optimizados": sorted(ensambles_opt, key=lambda n: -resultados_cv[n]["F1_mean"]),
}
orden_test = [n for g in grupos_test.values() for n in g]
tabla_test = pd.DataFrame({n: metricas(y_test, preds_test[n], scores_test[n]) for n in orden_test}).T

def clave_cv(nombre):
    """La regla con umbral comparte modelo (y F1_train) con la LR base; el resto usa su propio nombre."""
    return NOMBRE_LR if nombre == NOMBRE_LR_REGLA else nombre

# F1 de CV de la regla P(Dropout) >= UMBRAL_LR: mismos folds que evaluar_cv, sobre los puntajes OOF de la LR
score_lr_oof = oof[NOMBRE_LR][1]
f1_folds_regla = [f1_score(y_train[idx], predecir_regla(score_lr_oof[idx], UMBRAL_LR), average="macro")
                  for _, idx in cv.split(X_train, y_train)]
f1_cv = {n: resultados_cv[clave_cv(n)]["F1_mean"] for n in orden_test}
f1_cv[NOMBRE_LR_REGLA] = np.mean(f1_folds_regla)

tabla_test["F1 (CV)"] = [f1_cv[n] for n in orden_test]
tabla_test["Test − CV (F1)"] = tabla_test["F1"] - tabla_test["F1 (CV)"]
tabla_test["Predicción (ms)"] = [tiempos_pred[n] * 1000 for n in orden_test]

for titulo, nombres in grupos_test.items():
    print(titulo)
    display(tabla_test.loc[nombres].round(4))

print("Recall por clase en Test:")
display(pd.DataFrame({n: recall_score(y_test, preds_test[n], average=None) for n in orden_test}, index=CLASES).T.round(4))
print("F1 por clase en Test:")
display(pd.DataFrame({n: f1_score(y_test, preds_test[n], average=None) for n in orden_test}, index=CLASES).T.round(4))

guardar_tabla(tabla_test, "04_test_todos_los_modelos")
guardar_tabla(pd.DataFrame({n: recall_score(y_test, preds_test[n], average=None) for n in orden_test}, index=CLASES).T, "04_test_recall_por_clase")
guardar_tabla(pd.DataFrame({n: f1_score(y_test, preds_test[n], average=None) for n in orden_test}, index=CLASES).T, "04_test_f1_por_clase")

**Interpretación.**

- **En Test el orden de los ensambles es casi el contrario que en CV.** Por F1 macro, entre los 10 optimizados: Random Forest 0.7161, Bagging 0.7130, Pasting 0.7096, *Voting soft* (el elegido por CV) 0.7028, Stacking 0.6970, EBM 0.6967, *Voting hard* 0.6928, Gradient Boosting 0.6872, AdaBoost 0.6774 e HistGradientBoosting 0.6752. La correlación de Spearman entre el orden en CV y el orden en Test es de −0.32 (p = 0.37): **la CV no predijo qué ensamble sería mejor en Test.** Los tres que lideran en Test (Random Forest, Bagging y Pasting) ocupaban los lugares 7, 10 y 8 de los 10 en CV.
- **El F1 de Test es más bajo que el de CV en casi todos:** −0.031 en promedio en los 10 ensambles optimizados (de −0.059 en HistGradientBoosting a +0.003 en Bagging), casi lo mismo que la regresión logística (−0.029) y el SVM RBF (−0.032). Como el modelo que no se optimizó pierde lo mismo, **lo principal de la caída es que esta partición de Test es algo más difícil que el promedio de los folds**, no el sobreajuste de la búsqueda. Entre modelos la pérdida varía de 0.059 (HistGradientBoosting) y 0.045 (*Voting hard*) a 0.010 (Random Forest) y 0.009 (Pasting), una diferencia de 0.05 que, con intervalos de ±0.03, tampoco es concluyente.
- **La optimización no se transfiere a Test de forma consistente.** El F1 de CV subió en los 10 ensambles; en Test el F1 subió en 5 (*Voting soft*, Bagging, Pasting, Random Forest, EBM) y bajó en 4 (*Voting hard*, Gradient Boosting, HistGradientBoosting y Stacking) respecto a sus versiones base; AdaBoost quedó igual (0.6768 y 0.6774). Es lo que se espera de ganancias de unas milésimas en CV (Sección 21).
- **Los ensambles contra la referencia:** el SVM lineal optimizado (0.7078) supera en F1 macro a *Voting soft*; la regresión logística con su regla tiene 0.6832, el SVM RBF 0.6899. Todos los modelos, salvo el árbol único y el *Dummy*, están dentro de un rango de 0.041 de F1 macro, y los intervalos de la Sección 25 miden ±0.03: **ningún modelo es identificable como el mejor.** El árbol único (0.6257) es el más bajo con claridad.
- **Por clase, la dificultad es la misma de siempre:** F1 de *Dropout* entre 0.73 y 0.77, de *Graduate* entre 0.81 y 0.86 y de *Enrolled* entre 0.46 y 0.53 en los ensambles optimizados. El *recall* de *Enrolled* va de 40% (SVM RBF) a 67% (EBM), contra 57% de la regresión logística con regla y 53% de *Voting soft*; es decir, los modelos con más Accuracy (SVM, *Voting soft*) son los que menos reconocen a esa clase.

### 25. Intervalos de confianza y pruebas pareadas
Además de las métricas puntuales se reportan:
- **Intervalos de confianza de 95% por bootstrap** (2,000 remuestreos de los 885 estudiantes de Test) para la regresión logística, el SVM y los 10 ensambles optimizados.
- **Bootstrap pareado** de la diferencia contra la regresión logística: en cada remuestreo se evalúan ambos modelos sobre los mismos estudiantes.
- **Prueba de McNemar** (exacta): compara en cuántos estudiantes acierta uno y falla el otro.

In [ ]:
# Intervalos de confianza por bootstrap (2,000 remuestreos de los 885 estudiantes de Test)
def pct(a, q=(2.5, 97.5)):
    """Percentiles por columna ignorando NaN (el Voting hard no tiene ROC-AUC)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanpercentile(a, q, axis=0)

B = B_BOOT
rng = np.random.default_rng(RANDOM_STATE)
idx_boot = rng.integers(0, len(y_test), size=(B, len(y_test)))
modelos_boot = [NOMBRE_LR, NOMBRE_LR_REGLA, NOMBRE_SVM] + grupos_test["Ensambles optimizados"]
t0 = time.perf_counter()
def score_boot(n, i):
    """Puntajes del remuestreo i; None si el modelo no tiene puntaje continuo (Voting hard)."""
    return None if scores_test[n] is None else scores_test[n][i]

boot = {n: np.array([list(metricas(y_test[i], preds_test[n][i], score_boot(n, i)).values()) for i in idx_boot])
        for n in modelos_boot}
print(f"Bootstrap de {len(modelos_boot)} modelos: {time.perf_counter() - t0:.0f} s")

ic = {}
for n in modelos_boot:
    lo, hi = pct(boot[n])
    ic[n] = {m: (f"{tabla_test.loc[n, m]:.4f} [{l:.4f}, {h:.4f}]" if not np.isnan(tabla_test.loc[n, m]) else "—")
             for m, l, h in zip(METRICAS, lo, hi)}
print("Test: valor puntual [IC 95% bootstrap]")
display(pd.DataFrame(ic).T)

# Cada modelo contra la regresión logística con su regla (resultado oficial del Proyecto F_1): bootstrap pareado + McNemar
def comparar_pareado(a, b):
    d = boot[a] - boot[b]
    ok_a, ok_b = preds_test[a] == y_test, preds_test[b] == y_test
    n_ab, n_ba = int((ok_a & ~ok_b).sum()), int((~ok_a & ok_b).sum())
    p_mcnemar = stats.binomtest(n_ab, n_ab + n_ba, 0.5).pvalue if n_ab + n_ba > 0 else 1.0
    return d, n_ab, n_ba, p_mcnemar

def resumen_vs(ref):
    filas = {}
    for n in modelos_boot:
        if n in (ref, NOMBRE_LR, NOMBRE_LR_REGLA):
            continue
        d, n_ab, n_ba, p = comparar_pareado(n, ref)
        fila = {}
        for m in ["F1", "Recall", "Accuracy", "MCC", "ROC-AUC"]:
            j = METRICAS.index(m)
            if np.isnan(d[:, j]).all():
                fila[m] = "—"
                continue
            lo, hi = pct(d[:, j])
            fila[m] = f"{tabla_test.loc[n, m] - tabla_test.loc[ref, m]:+.4f} [{lo:+.3f}, {hi:+.3f}]"
        fila["McNemar (solo él / solo LR)"] = f"{n_ab} / {n_ba}  (p={p:.3f})"
        filas[n] = fila
    return pd.DataFrame(filas).T

print(f"Diferencia contra {NOMBRE_LR_REGLA}: valor [IC 95% bootstrap pareado]")
pareadas_regla = resumen_vs(NOMBRE_LR_REGLA)
display(pareadas_regla)
guardar_tabla(pareadas_regla, "04_pareadas_vs_lr_regla")   # insumo de reporte/ (tabla de comparaciones pareadas)
print(f"Diferencia contra {NOMBRE_LR} (regla por defecto)")
display(resumen_vs(NOMBRE_LR))
guardar_tabla(pd.DataFrame(ic).T, "04_test_intervalos_bootstrap")

In [ ]:
# Figura para la presentación: F1 macro y Accuracy en Test con su intervalo de confianza de 95% (bootstrap de los 885 estudiantes de Test)
ic_arr = {n: pct(boot[n]) for n in modelos_boot}
modelos_fig = sorted(modelos_boot, key=lambda n: tabla_test.loc[n, "F1"])
fig, axes = plt.subplots(1, 2, figsize=(15, 6.2), sharey=True)
for ax, m, color in zip(axes, ["F1", "Accuracy"], ["tab:blue", "tab:green"]):
    j = METRICAS.index(m)
    val = np.array([tabla_test.loc[n, m] for n in modelos_fig])
    lo = np.array([ic_arr[n][0][j] for n in modelos_fig])
    hi = np.array([ic_arr[n][1][j] for n in modelos_fig])
    colores = ["tab:red" if n in (NOMBRE_LR, NOMBRE_LR_REGLA) else color for n in modelos_fig]
    ax.barh(range(len(modelos_fig)), val, xerr=[val - lo, hi - val], color=colores, alpha=0.85, capsize=3)
    ax.set_yticks(range(len(modelos_fig)))
    ax.set_yticklabels([n.replace(" optimizado", "") for n in modelos_fig])
    ax.set_xlim(0.55, 0.82)
    ax.set_xlabel({"F1": "F1 macro en Test", "Accuracy": "Accuracy en Test"}[m] + " (barra de error: IC 95% bootstrap)")
    ax.set_title(f"{m} — Test (rojo: regresión logística)")
plt.tight_layout()
guardar_fig("04_test_f1_accuracy_con_ic")

**Interpretación.**

- **Los intervalos son anchos: ±0.03 en F1 macro y en Accuracy**, con 885 estudiantes. Por ejemplo, la regresión logística con regla tiene F1 0.683 [0.651, 0.712], *Voting soft* 0.703 [0.669, 0.734] y Random Forest 0.716 [0.684, 0.747]; los tres intervalos se traslapan casi por completo. El ROC-AUC tiene un intervalo algo más angosto (≈ ±0.019).
- **Contra la regresión logística con su regla** (resultado oficial del Proyecto F_1), solo Random Forest (+0.033 [0.006, 0.060]) y Bagging (+0.030 [0.001, 0.059]) tienen un intervalo de F1 macro que excluye el cero, y ambos son justo los dos modelos que **mejor salieron en Test**, es decir, elegidos *después* de ver los resultados. Con 10 modelos × 5 métricas se hacen unos 50 intervalos al 95%: es de esperar que alguno excluya el cero por azar. *Voting soft* (+0.020 [−0.005, 0.043]) y Pasting (+0.026 [−0.002, 0.054]) lo incluyen, y los otros siete modelos están a menos de 0.015 de ella.
- **En Accuracy y MCC la ventaja de varios modelos sí es clara:** SVM RBF (+0.037 en Accuracy), Random Forest (+0.034), Bagging (+0.036), Pasting (+0.029) y *Voting soft* (+0.031), todos con intervalos que no incluyen cero, y McNemar exacto con p = 0.002 a 0.005 (0.021 en Pasting). Son los modelos que reconocen mejor a *Graduate*, la clase más grande. Los p-valores no están corregidos por las 10 comparaciones; con una corrección de Holm los cuatro más pequeños quedarían alrededor de 0.02 a 0.05, es decir, en el límite.
- **En ROC-AUC, nadie supera a la regresión logística** (los intervalos de todos los ensambles incluyen cero, incluido *Voting soft*: +0.006 [−0.002, 0.014]). El SVM RBF es peor (−0.031 [−0.044, −0.018]) por el artefacto de `decision_function` descrito en la Sección 26.
- **Gradient Boosting, HistGradientBoosting, AdaBoost, EBM, Stacking y *Voting hard* son indistinguibles de la regresión logística** en F1 macro (diferencias de −0.008 a +0.014, todos los intervalos incluyen cero y McNemar con p entre 0.25 y 1.00).

### 26. Detalle del ensamble final, ROC-AUC del SVM y clase *Dropout*

In [ ]:
# Detalle del ensamble final contra la regresión logística con su regla: las 8 métricas
d, n_ab, n_ba, p = comparar_pareado(NOMBRE_FINAL, NOMBRE_LR_REGLA)
lo, hi = pct(d)
detalle = pd.DataFrame({"Diferencia (Test)": tabla_test.loc[NOMBRE_FINAL, METRICAS].values - tabla_test.loc[NOMBRE_LR_REGLA, METRICAS].values,
                        "IC 95% inferior": lo, "IC 95% superior": hi,
                        "% remuestreos con el ensamble mejor": (d > 0).mean(axis=0) * 100}, index=METRICAS)
print(f"{NOMBRE_FINAL} − {NOMBRE_LR_REGLA}")
display(detalle.round(4))
print(f"McNemar: acierta solo el ensamble en {n_ab} estudiantes, solo la LR en {n_ba} → p = {p:.4f}")

# ROC-AUC del SVM según el puntaje (con 3 clases, decision_function de SVC solo resume votos; ver sección de referencias)
svm_prob = clone(modelos_ajustados[NOMBRE_SVM]).set_params(model__probability=True, model__random_state=RANDOM_STATE)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)         # probability=True es obsoleto desde scikit-learn 1.9
    svm_prob.fit(X_train, y_train)
    proba_test_svm = svm_prob.predict_proba(X_test)
scores_test["SVM RBF optimizado (probabilidades)"] = proba_test_svm
puntajes = {"SVM — decision_function": scores_test[NOMBRE_SVM], "SVM — probabilidades (libsvm)": proba_test_svm,
            "LR (F_1) — probabilidades": scores_test[NOMBRE_LR], f"{NOMBRE_FINAL} — probabilidades": scores_test[NOMBRE_FINAL]}
auc_test = pd.DataFrame({n: [roc_auc_score(y_test == k, sc[:, k]) for k in range(len(CLASES))] for n, sc in puntajes.items()}, index=CLASES)
auc_test.loc["Macro (OvR)"] = auc_test.mean()
print("ROC-AUC en Test por clase según el puntaje:")
display(auc_test.round(4))

# Clase Dropout (la de interés para la alerta temprana): esa clase contra las otras dos
def metricas_dropout(y_true, y_pred, y_score):
    t, p_ = (y_true == 0), (y_pred == 0)
    return {"Precision": precision_score(t, p_, zero_division=0), "Recall": recall_score(t, p_),
            "F1": f1_score(t, p_), "ROC-AUC": roc_auc_score(t, y_score[:, 0]) if y_score is not None else np.nan}

cols_d = ["Precision", "Recall", "F1", "ROC-AUC"]
nombres_dropout = [NOMBRE_LR, NOMBRE_LR_REGLA, NOMBRE_SVM] + grupos_test["Ensambles optimizados"]
tabla_dropout = pd.DataFrame({n: metricas_dropout(y_test, preds_test[n], scores_test[n]) for n in nombres_dropout}).T
tabla_dropout[f"Detectados (de {(y_test == 0).sum()})"] = [int(((preds_test[n] == 0) & (y_test == 0)).sum()) for n in nombres_dropout]
tabla_dropout["Falsas alarmas"] = [int(((preds_test[n] == 0) & (y_test != 0)).sum()) for n in nombres_dropout]
tabla_dropout["Dropout → Graduate"] = [int(((preds_test[n] == 2) & (y_test == 0)).sum()) for n in nombres_dropout]
print("Clase Dropout en Test:")
display(tabla_dropout.round(4))
bd = {n: np.array([list(metricas_dropout(y_test[i], preds_test[n][i], scores_test[n][i]).values()) for i in idx_boot])
      for n in [NOMBRE_FINAL, NOMBRE_LR_REGLA]}
dif = bd[NOMBRE_FINAL] - bd[NOMBRE_LR_REGLA]
lo, hi = pct(dif)
print(f"Clase Dropout: {NOMBRE_FINAL} − {NOMBRE_LR_REGLA}")
display(pd.DataFrame({"Diferencia (Test)": tabla_dropout.loc[NOMBRE_FINAL, cols_d].values - tabla_dropout.loc[NOMBRE_LR_REGLA, cols_d].values,
                      "IC 95% inferior": lo, "IC 95% superior": hi,
                      "% remuestreos con el ensamble mejor": (dif > 0).mean(axis=0) * 100}, index=cols_d).round(4))
guardar_tabla(tabla_dropout, "04_test_clase_dropout")

**Interpretación.**

- **Ensamble final contra la regresión logística con regla:** mejora todas las métricas (de +0.006 en ROC-AUC a +0.038 en Kappa), pero el intervalo de 95% excluye el cero **solo en las métricas que ponderan por el tamaño de las clases**: Accuracy (+0.031 [0.010, 0.051]), F1 ponderado (+0.021 [0.002, 0.040]), MCC (+0.035 [0.004, 0.066]) y Kappa (+0.038 [0.006, 0.070]). En las métricas macro (F1 +0.020 [−0.005, 0.043], *recall* +0.012 [−0.013, 0.037], *precision* +0.018) el ensamble es mejor en 83% a 94% de los remuestreos, pero con un intervalo que cruza el cero. McNemar: acierta solo el ensamble en 55 estudiantes y solo la regresión logística en 28 (p = 0.004). **La ventaja es real en Accuracy y no demostrable en F1 macro.**
- **De dónde viene la Accuracy:** la regresión logística con regla manda 79 *Graduate* a *Enrolled* y el ensamble 46; el ensamble acierta 381 de 442 *Graduate* (86%) contra 348 (79%). Pierde algo de *Enrolled* (85 de 159 contra 91).
- **ROC-AUC del SVM:** con `decision_function` es 0.8529 y con las probabilidades de libsvm 0.8839, igual que la regresión logística (0.8838). La diferencia está casi toda en *Enrolled* (0.7255 contra 0.8136), la clase en que el puntaje de votos uno-contra-uno (OvO) resume peor el ordenamiento. Por eso la fila del SVM lleva la nota ᵈ en la tabla de literatura. *Voting soft* tiene 0.8900 y es el mejor de los cuatro puntajes comparados en cada clase (0.9151 en *Dropout*, 0.8213 en *Enrolled* y 0.9336 en *Graduate*), pero por menos de 0.01 sobre la regresión logística.
- **Clase *Dropout* (la de interés para una alerta temprana):** el ensamble y la regresión logística con regla **empatan**: detectan 203 de 284 (71.5%), con 51 y 50 falsas alarmas, F1 0.7546 contra 0.7561 (diferencia −0.001 [−0.024, 0.020]). Ningún ensamble detecta de forma apreciable más *Dropout*: de 186 (AdaBoost, 65.5%) a 204 (Pasting, 71.8%), contra 208 del SVM RBF, que a cambio tiene 55 falsas alarmas y manda 38 *Dropout* a *Graduate* (el error más costoso; Stacking solo 22). Random Forest tiene el mejor F1 de *Dropout* (0.7686) por ser el de mayor precisión (84.1%).

### 27. Curvas ROC y matrices de confusión en Test

In [ ]:
modelos_roc = [(NOMBRE_LR, "LR (F_1)"), ("SVM RBF optimizado (probabilidades)", "SVM RBF (prob.)"),
               ("Random Forest optimizado", "RF"), ("Gradient Boosting optimizado", "GB"),
               ("HistGradientBoosting optimizado", "HGB"), ("EBM optimizado", "EBM"),
               (NOMBRE_FINAL, NOMBRE_FINAL.replace(" optimizado", "")), ("Stacking optimizado", "Stacking")]
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
for ax, (k, clase) in zip(axes, enumerate(CLASES)):
    for (n, etiqueta), color in zip(modelos_roc, sns.color_palette("tab10", len(modelos_roc))):
        fpr, tpr, _ = roc_curve(y_test == k, scores_test[n][:, k])
        ax.plot(fpr, tpr, lw=1.8, color=color, label=f"{etiqueta} (AUC={roc_auc_score(y_test == k, scores_test[n][:, k]):.3f})")
    ax.plot([0, 1], [0, 1], color=COLOR_GRIS, lw=1)
    ax.set_title(f"ROC en Test — {clase} contra el resto")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
guardar_fig("04_test_roc")

mostrar = [NOMBRE_LR_REGLA, NOMBRE_SVM, "Random Forest optimizado", "HistGradientBoosting optimizado", NOMBRE_FINAL, "EBM optimizado"]
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
for ax, nombre in zip(axes.ravel(), mostrar):
    ConfusionMatrixDisplay(confusion_matrix(y_test, preds_test[nombre]), display_labels=CLASES).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Test — {nombre}", fontsize=10.5)
    ax.set_xlabel("Clase predicha")
    ax.set_ylabel("Clase real")
    ax.grid(False)
plt.tight_layout()
guardar_fig("04_test_matrices_confusion")

**Interpretación.**

- **Curvas ROC:** las curvas de los 8 modelos casi se superponen en cada clase. El AUC por clase varía poco entre modelos: 0.902 a 0.915 en *Dropout*, 0.798 a 0.822 en *Enrolled* y 0.925 a 0.934 en *Graduate*. HistGradientBoosting es el más bajo en *Dropout* y en *Enrolled*. **La clase *Enrolled* es la peor ordenada (AUC ≈ 0.82)**, aunque todos los modelos la ordenan mucho mejor que el azar; parte de la dificultad con *Enrolled* viene del punto de operación y de la información disponible, no solo del modelo.
- **Matrices de confusión:** el SVM RBF es el que más *Graduate* acierta (403 de 442, 91%) y el que menos *Enrolled* (64 de 159, 40%); el EBM es el opuesto (106 *Enrolled*, 67%, y 346 *Graduate*, 78%). Random Forest (97 y 374) y la regresión logística con regla (91 y 348) quedan entre los dos extremos y *Voting soft* (85 y 381) más cerca del SVM. *Dropout* se detecta de forma parecida en todos (195 a 208 de 284).
- **Los errores se concentran en *Enrolled*:** entre tres cuartas partes y cuatro quintas partes de los errores de cada matriz involucran a esa clase (74% en el SVM, 79% en *Voting soft*, 84% en la regresión logística con regla); entre 33% (EBM) y 60% (SVM) de los 159 *Enrolled* se clasifican mal. Los *Dropout* se confunden más con *Enrolled* (51 a 65 en cinco de los seis modelos; 38 en el SVM) que con *Graduate* (24 a 38).

**Preguntas de discusión**

1. **¿Qué ensamble funcionó mejor?** Depende de con qué se mida, y la diferencia entre ellos es pequeña. Por el criterio fijado antes (F1 macro en CV de 5 folds) fue *Voting soft* (0.7375 ± 0.0224); en la CV repetida, el mismo (0.7341); en Test, el mejor F1 fue Random Forest (0.7161) y *Voting soft* quedó cuarto (0.7028). El rango entre los 10 ensambles es de 0.027 en CV y de 0.041 en Test, menos que lo que dan los intervalos (±0.03). **No hay un mejor ensamble identificable con estos datos.**
2. **¿Un ensamble supera a un modelo individual?** Frente a un árbol de decisión, sí y por mucho (F1 macro en CV de 0.656 a 0.71–0.74; en Test de 0.626 a 0.68–0.72), porque promediar muchos árboles reduce la varianza que domina a un árbol solo. Frente a la regresión logística o al SVM, la ventaja es pequeña: +0.016 de F1 macro para el mejor en CV repetida (no significativa tras Holm) y +0.020 en Test (IC que incluye el cero), aunque sí significativa en Accuracy (+0.031 en Test, McNemar p = 0.004) y MCC.
3. **¿*Voting hard* o *soft*?** Empatan en CV (0.7373 y 0.7375). El *soft* aprovecha las probabilidades: mejor Accuracy en CV (0.7858 contra 0.7793), en Test (0.7559 contra 0.7390) y ROC-AUC (0.8900 en Test), y es el único de los dos con ventaja significativa en McNemar contra la regresión logística (p = 0.004 contra 0.25). El *hard* no puede producir puntajes ni ROC-AUC.
4. **¿Bagging, Pasting o Random Forest?** En CV: Random Forest 0.7262, Pasting 0.7189 y Bagging 0.7101; sin diferencia sostenible (desviación ≈ 0.02). En Test los tres son los mejores ensambles (0.7161, 0.7096 y 0.7130). La estimación OOF coincide con la CV (Secciones 11.1 y 12.1), por lo que en conjuntos de este tamaño es una alternativa gratuita a la validación cruzada.
5. **¿Boosting o *bagging*?** En CV el boosting es lo mejor de una sola familia (HistGradientBoosting 0.7340 y Gradient Boosting 0.7316, contra 0.7101 a 0.7262), pero con los mayores gaps (0.185 a 0.213) y en Test quedó por debajo (0.6752 y 0.6872 contra 0.7096 a 0.7161). Las curvas por etapas (Sección 18) muestran que el techo es el mismo para cualquier `learning_rate`, y que lo que cambia es cuántos árboles hacen falta.
6. **¿Stacking mejora a *Voting*?** No. 0.7331 de F1 en CV contra 0.7375, y es 4 veces más lento de ajustar (≈ 26 s contra ≈ 6 s) porque reentrena sus modelos base en cada fold interno.
7. **¿Qué aporta el EBM?** Un F1 comparable (0.7309 en CV) con el menor sobreajuste (gap 0.064) y el mayor *recall* de *Enrolled* en CV y en Test (65.0% OOF y 67% en Test), siendo además interpretable (Sección 29). Su costo es el tiempo de ajuste y que con 3 clases no se pueden graficar sus interacciones.
8. **¿Qué hiperparámetros importaron?** La profundidad de los árboles en AdaBoost (F1 de 0.678 con profundidad 2 a 0.718 con 6) y `min_samples_leaf` en Random Forest (de 0.725 con hojas de 1 a 0.689 con hojas de 40, con el gap bajando de 0.27 a 0.02). `max_features` y el número de árboles en los bosques importaron poco, y en los boosting `learning_rate` y `n_estimators` se compensan. En el EBM, el número de interacciones (el GAM puro es 0.017 peor).
9. **¿Hubo sobreajuste?** Sí, en el sentido de entrenamiento: los gaps train − CV de los ensambles son de 0.06 a 0.21 (la regresión logística, 0.033). Pero la pérdida al pasar a Test (−0.031 de F1 en promedio) es la misma que la de los modelos simples (−0.029 y −0.032), y los más sobreajustados en entrenamiento no son sistemáticamente los que más pierden (Spearman entre el gap y la pérdida: +0.20, p = 0.58). El gap alto de los bosques sobreestima el daño porque cada árbol predice los datos de entrenamiento que vio.
10. **¿Vale la pena el ensamble?** Para F1 macro, no hay evidencia. Para Accuracy y MCC, sí, con modelos como *Voting soft* o Random Forest. El costo: *Voting soft* tarda ≈ 6 s en ajustarse (contra ≈ 2 s de la regresión logística y 0.2 s del SVM RBF) y pierde la interpretabilidad. Si lo que importa es la Accuracy, el ensamble compensa; si importa *Enrolled* o *Dropout*, la regresión logística y el EBM rinden parecido con mucha menos complejidad.

## Fase 9. Comparación Train/CV vs. Test

### 28. Análisis de generalización

In [ ]:
nombres_gen = [NOMBRE_LR, NOMBRE_SVM] + grupos_test["Ensambles optimizados"]
gen = pd.DataFrame({n: {"F1 train": resultados_cv[n]["F1_train"], "F1 CV (5 folds)": resultados_cv[n]["F1_mean"],
                        "F1 Test": tabla_test.loc[n, "F1"], "Gap train − CV": resultados_cv[n]["gap_F1"],
                        "Test − CV (F1)": tabla_test.loc[n, "F1"] - resultados_cv[n]["F1_mean"],
                        "Test − CV (Accuracy)": tabla_test.loc[n, "Accuracy"] - resultados_cv[n]["Accuracy_mean"],
                        "Test − CV (MCC)": tabla_test.loc[n, "MCC"] - resultados_cv[n]["MCC_mean"],
                        "Test − CV (ROC-AUC)": tabla_test.loc[n, "ROC-AUC"] - resultados_cv[n]["ROC-AUC_mean"]}
                    for n in nombres_gen}).T
display(gen.round(4))
fig, ax = plt.subplots(figsize=(10, 5.5))
orden = gen.sort_values("F1 Test").index
y_pos = np.arange(len(orden))
ax.scatter(gen.loc[orden, "F1 train"], y_pos, marker="s", color="tab:gray", label="train")
ax.scatter(gen.loc[orden, "F1 CV (5 folds)"], y_pos, color="tab:blue", label="CV (5 folds)")
ax.scatter(gen.loc[orden, "F1 Test"], y_pos, marker="D", color="tab:red", label="Test")
ax.set_yticks(y_pos)
ax.set_yticklabels([n.replace(" optimizado", "") for n in orden])
ax.set_xlabel("F1 macro")
ax.set_title("Train, CV y Test: F1 macro por modelo")
ax.legend()
plt.tight_layout()
guardar_fig("04_train_cv_test_f1")
guardar_tabla(gen, "04_generalizacion")

**Interpretación.**

- **En todos los modelos el orden es Train > CV > Test (salvo Bagging en F1), con una jerarquía clara de sobreajuste *en entrenamiento*:** regresión logística 0.750 de F1 en Train (gap 0.033), SVM RBF 0.765 (0.043), EBM 0.795 (0.064), AdaBoost 0.804 (0.087), Stacking 0.881 (0.148), Bagging 0.881 (0.171) y 0.912 a 0.947 (gaps de 0.178 a 0.213) en *Voting*, Gradient Boosting, Pasting, Random Forest e HistGradientBoosting.
- **Pero el gap de entrenamiento casi no predice la pérdida en Test** (Spearman de +0.20, p = 0.58, entre el gap y la pérdida en los 10 ensambles). Random Forest (gap 0.203) pierde solo 0.010 de F1, Pasting (0.193) 0.009 y Bagging (0.171) gana 0.003, mientras que el EBM (gap 0.064) pierde 0.034, AdaBoost (0.087) 0.040 e HistGradientBoosting (0.213) 0.059. En un bosque, el F1 de entrenamiento está inflado porque cada árbol predice observaciones que vio al construirse (en promedio cada observación está en 63% de los árboles); ese gap exagera el daño, y por eso la comparación se hace con CV y Test.
- **La pérdida de CV a Test es de −0.031 de F1 en promedio** (−0.030 de Accuracy, −0.040 de MCC y −0.007 de ROC-AUC), casi igual a la de la regresión logística (−0.029) y el SVM RBF (−0.032), que no pasaron por una búsqueda en esta práctica. Que el ROC-AUC baje mucho menos que las demás métricas indica que **el ordenamiento de los estudiantes generaliza mejor que las decisiones de clasificación**. Con un error estándar de ≈ 0.016 en el F1 de Test (Sección 25), una caída de 0.03 es compatible con que esta partición sea un poco más difícil, algo que afecta a todos los modelos por igual.
- **Los que lideraban en CV son los que más caen en el orden.** *Voting soft* (1º en CV) queda 4º en Test, *Voting hard* (2º) queda 7º e HistGradientBoosting (3º) queda último; Bagging (último en CV) y Pasting suben a los primeros lugares. Es lo que se espera de la *maldición del ganador* de la Sección 23 (los mejores resultados de CV son en parte suerte), aunque con 10 modelos y diferencias tan pequeñas no se puede afirmar que sea más que ruido (Spearman entre los dos órdenes: −0.32, p = 0.37).

## Fase 10. Interpretabilidad

### 29. ¿Qué variables usan los modelos?
**Random Forest.** La importancia por impureza (*MDI*, la caída promedio de impureza que aporta cada variable) se calcula sobre Train y favorece a las variables con muchas categorías o valores distintos y reparte la importancia entre variables correlacionadas. La **importancia por permutación** mide cuánto cae el F1 macro en Test al barajar una variable; es más confiable, pero también se reparte entre variables correlacionadas. Las columnas *one-hot* se suman por variable original. Se calcula también para el ensamble final.

**EBM.** La importancia de cada término es la media del valor absoluto de su aporte (en log-odds) sobre Train; las **funciones de forma** muestran cuánto suma o resta cada variable a la log-odds de cada clase según su valor (la banda es la variabilidad entre los 8 *bags*). Como el modelo se entrenó con pesos balanceados, el nivel absoluto de las curvas está desplazado; lo que se interpreta es su **forma y pendiente**.

In [ ]:
# Random Forest optimizado: importancia por impureza (MDI) contra importancia por permutación en Test
def variable_original(nombre_col):
    """Agrupa las columnas one-hot en la variable nominal de la que vienen."""
    n = nombre_col.split("__", 1)[1]
    for c in nominal_cols:
        if n.startswith(c + "_"):
            return c
    return n

rf_final = modelos_ajustados["Random Forest optimizado"]
cols_prep = list(rf_final.named_steps["preprocessor"].get_feature_names_out())
mdi_cols = pd.Series(rf_final.named_steps["model"].feature_importances_, index=cols_prep)
mdi = mdi_cols.groupby([variable_original(c) for c in cols_prep]).sum()

perm_rf = permutation_importance(rf_final, X_test, y_test, scoring="f1_macro", n_repeats=10,
                                 random_state=RANDOM_STATE, n_jobs=N_JOBS)
perm_final = permutation_importance(modelos_ajustados[NOMBRE_FINAL], X_test, y_test, scoring="f1_macro", n_repeats=5,
                                    random_state=RANDOM_STATE, n_jobs=N_JOBS)
imp = pd.DataFrame({"MDI (RF)": mdi, "Permutación RF (ΔF1)": pd.Series(perm_rf.importances_mean, index=X_test.columns),
                    f"Permutación {NOMBRE_FINAL.replace(' optimizado', '')} (ΔF1)": pd.Series(perm_final.importances_mean, index=X_test.columns)})
imp_std = pd.Series(perm_rf.importances_std, index=X_test.columns)

top = 12
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
mdi.sort_values().tail(top).plot.barh(ax=axes[0], color="tab:blue")
axes[0].set_title(f"MDI (impureza) del Random Forest — top {top}")
orden_perm = imp["Permutación RF (ΔF1)"].sort_values().tail(top).index
axes[1].barh(orden_perm, imp.loc[orden_perm, "Permutación RF (ΔF1)"], xerr=imp_std[orden_perm], color="tab:orange")
axes[1].set_title(f"Permutación en Test (caída de F1 macro) — top {top}")
plt.tight_layout()
guardar_fig("04_importancias_random_forest")

print("Top 12 por importancia de permutación del Random Forest (la variable se baraja en Test):")
display(imp.sort_values("Permutación RF (ΔF1)", ascending=False).head(12).round(4))
print(f"Correlación de Spearman entre MDI y permutación (36 variables): "
      f"{stats.spearmanr(imp['MDI (RF)'], imp['Permutación RF (ΔF1)']).statistic:.3f}")
guardar_tabla(imp, "04_importancia_variables")

In [ ]:
# Explainable Boosting Machine. Con 3 clases, interpret no puede graficar los pares de interacciones, así que:
#  - la importancia de los términos se toma del EBM optimizado (incluye interacciones, pero solo como número);
#  - las funciones de forma se grafican con el EBM sin interacciones (GAM puro), que se puede leer por completo.
ebm_gam = modelos_ajustados["EBM sin interacciones (GAM)"].named_steps["model"]
ebm_opt = modelos_ajustados["EBM optimizado"].named_steps["model"]
g_gam = ebm_gam.explain_global()
imp_gam = pd.Series(ebm_gam.term_importances(), index=ebm_gam.term_names_).sort_values(ascending=False)
imp_ebm_opt = pd.Series(ebm_opt.term_importances(), index=ebm_opt.term_names_).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))
for ax, (imp_t, titulo) in zip(axes, [(imp_gam, "EBM sin interacciones (GAM)"), (imp_ebm_opt, "EBM optimizado")]):
    sel = imp_t.head(15).iloc[::-1]
    ax.barh(sel.index, sel.values, color=["tab:orange" if " & " in n else "tab:blue" for n in sel.index])
    ax.set_xlabel("Importancia (media del |aporte| sobre Train, en log-odds)")
    ax.set_title(f"{titulo} — top 15 términos (naranja = interacción)")
plt.tight_layout()
guardar_fig("04_ebm_importancia_terminos")
n_inter = sum(" & " in n for n in imp_ebm_opt.index)
print(f"EBM optimizado: {len(imp_ebm_opt)} términos ({n_inter} interacciones). "
      f"Las interacciones suman {imp_ebm_opt[[' & ' in n for n in imp_ebm_opt.index]].sum() / imp_ebm_opt.sum():.1%} de la importancia total.")
display(pd.DataFrame({"GAM": imp_gam.head(10), "EBM optimizado": imp_ebm_opt.head(10)}).round(4))

def graficar_termino(ax, g, modelo, nombre):
    """Función de forma de un término principal: una curva (o grupo de barras) por clase, en log-odds."""
    d = g.data(modelo.term_names_.index(nombre))
    x, S = np.asarray(d["names"]), np.asarray(d["scores"], dtype=float)
    lo, hi = np.asarray(d["lower_bounds"], dtype=float), np.asarray(d["upper_bounds"], dtype=float)
    try:
        xf = x.astype(float)
        continuo = len(xf) == len(S) + 1            # x = bordes de los bins
    except (ValueError, TypeError):
        continuo = False
    for k, clase in enumerate(CLASES):
        if continuo:
            ax.step(xf, np.r_[S[:, k], S[-1, k]], where="post", color=PALETA_CLASES[clase], label=clase)
            ax.fill_between(xf, np.r_[lo[:, k], lo[-1, k]], np.r_[hi[:, k], hi[-1, k]], step="post", alpha=0.15,
                            color=PALETA_CLASES[clase])
        else:
            pos = np.arange(len(S)) + (k - 1) * 0.27
            ax.bar(pos, S[:, k], width=0.27, color=PALETA_CLASES[clase], label=clase,
                   yerr=[S[:, k] - lo[:, k], hi[:, k] - S[:, k]], error_kw=dict(lw=0.6))
            ax.set_xticks(np.arange(len(S)))
            ax.set_xticklabels([str(v) for v in x[:len(S)]], rotation=90, fontsize=7)
    ax.axhline(0, color=COLOR_GRIS, lw=0.8)
    ax.set_title(nombre, fontsize=10)
    ax.set_ylabel("aporte (log-odds)")

principales = [n for n in imp_gam.index if " & " not in n][:6]
fig, axes = plt.subplots(2, 3, figsize=(17, 8.5))
for ax, nombre in zip(axes.ravel(), principales):
    graficar_termino(ax, g_gam, ebm_gam, nombre)
axes[0, 0].legend(fontsize=8)
plt.suptitle("EBM sin interacciones — funciones de forma de los 6 términos más importantes (una curva por clase)", y=1.01)
plt.tight_layout()
guardar_fig("04_ebm_funciones_de_forma")

**Interpretación.**

- **Importancia por impureza (MDI) contra permutación en el Random Forest.** Los dos métodos coinciden en la cima (*Curricular units 2nd sem (approved)*: MDI 0.127, la mayor, y caída de F1 en Test de 0.072 al barajarla; *Tuition fees up to date*: 0.041 y 0.042; *1st sem (approved)*: 0.076 y 0.027), pero **discrepan en el resto: la correlación de Spearman entre los dos rankings de las 36 variables es de 0.56.** *Curricular units 2nd sem (grade)* es la segunda por MDI (0.087) y su permutación solo cuesta 0.021; *Course* tiene 0.050 de MDI y 0.013 de permutación; *Curricular units 1st sem (evaluations)*, 0.040 y 0.005. El MDI se calcula con Train, con árboles muy profundos (el F1 de entrenamiento es 0.93) y favorece a las variables continuas o con muchas categorías, que ofrecen más puntos de corte aun cuando no generalizan: *Father's occupation*, *Mother's occupation* y *Application mode* están en el top 12 por MDI, pero no en el de permutación. **La permutación en Test mide lo que el modelo realmente usa para predecir datos nuevos, y es la que se debe leer.**
- **El ensamble final depende todavía más de pocas variables.** Barajar *Curricular units 2nd sem (approved)* baja el F1 macro de *Voting soft* en 0.189 (0.072 en el Random Forest), seguida de *1st sem (approved)* (0.056) y *Tuition fees up to date* (0.045). Las demás variables de la tabla aportan menos de 0.015 cada una. El rendimiento del ensamble descansa en el avance académico del primer año y en si la colegiatura está al corriente.
- **EBM (importancia de los términos).** El EBM optimizado tiene 46 términos (36 variables y 10 interacciones) y las interacciones suman 19.1% de la importancia total, la mayor de ellas *Tuition fees up to date & GDP* (0.109). Los términos principales son los mismos que en el GAM puro: *Curricular units 2nd sem (approved)* (0.512 en el EBM y 0.602 en el GAM), *Course* (0.249), *Tuition fees up to date* (0.237) y *1st sem (approved)* (0.192). Las interacciones restan importancia a los términos principales (de 0.602 a 0.512 en el principal) y casi no cambian el orden (solo *Course* y *Tuition fees up to date* intercambian lugar).
- **Funciones de forma (GAM).** Con 0 unidades aprobadas en el 2º semestre, el aporte a la log-odds de *Dropout* es +1.2 y el de *Graduate* −1.4; con 20 aprobadas, −1.9 y +2.7, con un cruce cerca de 4 a 5 unidades. Con la colegiatura sin pagar, *Dropout* suma +1.1 y *Graduate* −1.4, contra −0.15 y +0.2 con la colegiatura al corriente. Con más evaluaciones presentadas en el 2º semestre, el aporte a *Enrolled* sube de −0.65 a +0.75 y el de *Graduate* baja, consistente con estudiantes que se presentan a muchos exámenes sin aprobarlos. Las bandas son anchas en los extremos (pocos estudiantes) y en los cursos con pocos matriculados, así que solo se interpretan las curvas en las regiones con datos.
- **Todos los métodos coinciden en las mismas variables** (unidades aprobadas en ambos semestres, colegiatura al corriente, curso, evaluaciones y calificaciones del 2º semestre), y coincide con lo que Praptomo et al. (2026) encontraron con SHAP: el avance académico del primer año pesa más que las características de ingreso. Son asociaciones predictivas, no causas. Y como todas las variables dominantes se conocen *después* de cursar uno o dos semestres, estos modelos no sirven como alerta de ingreso, sino a partir del final del primer semestre.

## Fase 11. Comparación con trabajos relacionados
### 30. Comparación externa

Se agregan a la tabla del Proyecto F_1 y de la práctica de SVM los modelos de este trabajo y los modelos de ensamble publicados con este dataset por Villar y de Andrade (2024). Todas las filas de este trabajo son en Test (885 estudiantes, la misma composición que el Test de Praptomo et al.). Precision, Recall y F1-Score son promedios **macro** de las 3 clases; ROC-AUC es uno contra el resto, macro. "—" indica que el trabajo no reporta la métrica. La tabla se genera con código: la marca ★ en ámbar es el mejor modelo de este trabajo según F1 macro, la ★ azul es el mejor de la literatura y ◆ es el ensamble final elegido con CV.

In [ ]:
# ---- Tabla comparativa con la literatura: 5 métricas (valores en Test) ----
CRITERIO_MEJOR = "F1"        # métrica con la que se marca "el mejor" de este trabajo y de la literatura (F1 macro)
cols_metricas = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
ND = "—"                     # no reportado por el estudio

# Literatura: valor tal como lo reportan los autores (texto) y marca de nota al pie
def fila_lit(estudio, datos, modelo, acc=None, prec=None, rec=None, f1=None, auc=None):
    return dict(Estudio=estudio, Datos=datos, Modelo=modelo, Accuracy=acc, Precision=prec, Recall=rec, F1=f1, **{"ROC-AUC": auc})

P, M, V = "Praptomo et al. (2026)", "Martins et al. (2023)", "Villar & de Andrade (2024)"
DP = "UCI 697, 4,424; Test de 885"
literatura = [
    fila_lit(P, DP, "Regresión Logística", ("0.7684",), ("0.7070",), ("0.6754",), ("0.6826",)),
    fila_lit(P, DP, "Random Forest", ("0.7718",), ("0.7275",), ("0.6859",), ("0.6979",)),
    fila_lit(P, DP, "XGBoost optimizado", ("0.7605",), ("0.7075",), ("0.6958",), ("0.7008",)),
    fila_lit(M, "Versión previa (4,433); fin del 1er semestre", "SVMSMOTE + Random Forest", rec=("0.749", "ᵃ"), f1=("0.707", "ᵇ")),
    fila_lit(M, "Fin del 1er semestre", "Balanced Random Forest", rec=("0.739", "ᵃ"), f1=("0.713", "ᵇ")),
    fila_lit(M, "Fin del 2º semestre, sin los que ya abandonaron", "SVMSMOTE + Random Forest", rec=("0.748", "ᵃ"), f1=("0.641", "ᵇ")),
    fila_lit(V, "UCI 697, con SMOTE; partición no reportada", "SVM", f1=("0.743", "ᶜ")),
    fila_lit(V, "ídem", "Random Forest", f1=("0.840", "ᶜ")),
    fila_lit(V, "ídem", "Gradient Boosting", f1=("0.813", "ᶜ")),
    fila_lit(V, "ídem", "XGBoost", f1=("0.857", "ᶜ")),
    fila_lit(V, "ídem", "CatBoost", f1=("0.850", "ᶜ")),
    fila_lit(V, "ídem", "LightGBM", f1=("0.853", "ᶜ")),
]
vals_lit = [float(f[CRITERIO_MEJOR][0]) if f[CRITERIO_MEJOR] else None for f in literatura]
i_mejor_lit = int(np.nanargmax([np.nan if v is None else v for v in vals_lit]))

filas = []
for i, f in enumerate(literatura):
    fila = {"Estudio": f["Estudio"], "Datos": f["Datos"], "Modelo": f["Modelo"] + (" ★" if i == i_mejor_lit else "")}
    for m in cols_metricas:
        fila[m] = ND if f[m] is None else "".join(f[m])
    filas.append(fila)

# Este trabajo (de tabla_test): referencias y ensambles optimizados
MODELOS_LIT = grupos_test["Referencias (Dummy, regresión logística, SVM)"][1:] + grupos_test["Ensambles optimizados"]
nuestros = tabla_test.loc[MODELOS_LIT]
mejor = nuestros[CRITERIO_MEJOR].idxmax()          # ignora NaN
fmt = lambda v: ND if pd.isna(v) else f"{v:.4f}"
for nombre, r in nuestros.iterrows():
    marca = (" ★" if nombre == mejor else "") + (" ◆" if nombre == NOMBRE_FINAL else "")
    filas.append({"Estudio": "Este trabajo", "Datos": "UCI 697; Test de 885", "Modelo": nombre + marca,
                  **{m: fmt(r[m]) + ("ᵈ" if (m == "ROC-AUC" and nombre.startswith("SVM") and not pd.isna(r[m])) else "")
                     for m in cols_metricas}})
tabla_lit = pd.DataFrame(filas)

def negrita(fila):
    if "★" in fila["Modelo"]:
        color = "255, 193, 7" if fila["Estudio"] == "Este trabajo" else "66, 165, 245"      # ámbar = nuestro, azul = literatura
        return [f"font-weight: bold; background-color: rgba({color}, 0.30)"] * len(fila)
    return ["font-weight: bold" if fila["Estudio"] == "Este trabajo" else ""] * len(fila)

display(tabla_lit.style.hide(axis="index").apply(negrita, axis=1).set_properties(subset=cols_metricas, **{"text-align": "right"}))
print(f"{ND} = métrica no reportada por el estudio.")
print(f"★ (ámbar) = mejor modelo de este trabajo en Test según {CRITERIO_MEJOR}: {mejor} ({nuestros.loc[mejor, CRITERIO_MEJOR]:.4f}). "
      "Es una marca descriptiva: la selección del modelo se hizo antes, con CV.")
print(f"◆ = ensamble final elegido con CV (F1 macro): {NOMBRE_FINAL} ({nuestros.loc[NOMBRE_FINAL, CRITERIO_MEJOR]:.4f} en Test).")
l = literatura[i_mejor_lit]
print(f"★ (azul)  = mejor modelo de la literatura según {CRITERIO_MEJOR}: {l['Modelo']} — {l['Estudio']} ({l[CRITERIO_MEJOR][0]}); ver nota ᶜ.")
guardar_tabla(tabla_lit, "04_comparacion_literatura", indice=False)

ᵃ Martins et al. reportan *balanced accuracy*, que en multiclase es exactamente el Recall macro.

ᵇ Martins et al. reportan un "F1 global" ponderado (0.745, 0.744 y 0.741). Aquí se muestra el F1 macro, calculado como el promedio de sus F1 por clase (Tablas 7 y 8 del artículo). Sus valores son el promedio de 10 repeticiones.

ᶜ Villar & de Andrade solo reportan el F1 por clase de cada modelo (por ejemplo, 0.79 *Graduate*, 0.76 *Dropout* y 0.68 *Enrolled* para el SVM); aquí se muestra su promedio. No reportan la proporción de la partición, el número de folds ni los hiperparámetros, y aplican SMOTE. Mencionan un "SMOTE test dataset" de 1,325 instancias: 1,325 ≈ 20% de 6,627 = 3 × 2,209, el tamaño del dataset después de sobremuestrear cada clase hasta el tamaño de *Graduate*. Esto sugiere que el SMOTE se aplicó **antes** de partir los datos, así que su Test contendría estudiantes sintéticos generados a partir de los de entrenamiento.

ᵈ ROC-AUC calculado con `decision_function`, que en `SVC` multiclase subestima el ordenamiento (Sección 26). Con probabilidades de libsvm, el SVM RBF optimizado tiene ROC-AUC = 0.8839 en Test.

**¿Son equivalentes los protocolos?**
- **Praptomo et al. (2026)** es la comparación más directa: mismo dataset, misma composición de Test (284 *Dropout*, 159 *Enrolled*, 442 *Graduate*).
  - Los ensambles optimizados de este trabajo tienen F1 macro de 0.675 a 0.716 en Test, contra 0.683 de su regresión logística, 0.698 de su Random Forest y 0.701 de su XGBoost. El ensamble final (*Voting soft*, 0.703) y los mejores en Test (Random Forest 0.716, Bagging 0.713 y Pasting 0.710) quedan por encima de su XGBoost por 0.002 a 0.015, pero con intervalos de ±0.03 (Sección 25), **no hay diferencia que se pueda sostener.**
  - Su Accuracy (0.761 a 0.772) queda en el extremo alto del rango de nuestros ensambles optimizados (0.714 a 0.762), pero detectan menos *Enrolled* (su XGBoost, 46.5%; su regresión logística, 33.3%) que los 40% (SVM RBF) a 67% (EBM) de esta práctica.
  - En *Dropout*, su XGBoost detecta 212 de 284 (74.7%), 4 más que nuestro mejor modelo (SVM RBF, 208) y 9 más que el ensamble final y la regresión logística con regla (203).
- **Martins et al. (2023):** mismas salvedades que en el Proyecto F_1 (otra versión de los datos con otra distribución de clases, partición 90/10 con 10 repeticiones, eliminación de los que ya abandonaron en la fase del 2º semestre). Su mejor modelo usa solo la información del primer semestre, y los de esta práctica usan ambos.
- **Villar & de Andrade (2024)** reportan F1 macro de 0.81 a 0.86 para los ensambles de árboles (Random Forest 0.840, XGBoost 0.857), de 0.12 a 0.14 por encima de nuestro Random Forest (0.716) en **el mismo dataset**. Esa distancia es varias veces mayor que cualquier diferencia entre modelos aquí, y es lo que se esperaría si el SMOTE se aplicó antes de la partición. Además, solo reportan el F1 por clase, sin Accuracy, Precision, Recall, AUC ni protocolo. Con esa información, el número no se puede comparar ni reproducir.
- **Ningún trabajo reporta MCC, Kappa, intervalos de confianza ni pruebas de significancia.** Con un solo valor puntual, una diferencia de 0.01 a 0.02 en F1 macro, como las que separan a casi todas las filas de esta tabla, no permite decir que un modelo sea mejor que otro.

# IV. Resultados y discusión

In [ ]:
print("Resumen en CV (5 folds, media ± desviación) de los modelos finales:")
finales = [NOMBRE_LR, "SVM RBF optimizado"] + sorted(ensambles_opt, key=lambda n: -resultados_cv[n]["F1_mean"])
display(tabla_resumen(finales)[["Accuracy", "Recall", "F1", "MCC", "ROC-AUC", "Gap F1 (train−CV)", "Ajuste con todo Train (s)"]])
print("Mejores hiperparámetros de cada búsqueda:")
display(pd.DataFrame({n: {"Hiperparámetros": {k.replace("model__", ""): v for k, v in s.best_params_.items()},
                          "F1 macro (CV)": round(s.best_score_, 4), "Búsqueda (s)": round(tiempos_busqueda[n])}
                      for n, s in searches.items() if "mapa de calor" not in n}).T.style.set_properties(**{"text-align": "left"}))
print(f"\nEnsamble final (criterio: F1 macro en CV): {NOMBRE_FINAL}")
print("Test:")
display(tabla_test.loc[[NOMBRE_LR_REGLA, NOMBRE_LR, NOMBRE_SVM, NOMBRE_FINAL], METRICAS].round(4))

### Análisis de errores (Test)

In [ ]:
real = pd.Categorical.from_codes(y_test, CLASES)
pred_f = pd.Categorical.from_codes(preds_test[NOMBRE_FINAL], CLASES)
pred_l = pd.Categorical.from_codes(preds_test[NOMBRE_LR_REGLA], CLASES)
print(f"Matriz de confusión en Test — {NOMBRE_FINAL} (izquierda) y LR con su regla (derecha):")
display(pd.concat({"Ensamble final": pd.crosstab(real, pred_f, rownames=["real"], colnames=["predicha"]),
                   "LR + regla 0.40": pd.crosstab(real, pred_l, rownames=["real"], colnames=["predicha"])}, axis=1))

ok_f, ok_l = preds_test[NOMBRE_FINAL] == y_test, preds_test[NOMBRE_LR_REGLA] == y_test
analisis = X_test.copy()
analisis["real"] = real
analisis["acierto"] = np.select([ok_f & ok_l, ok_f & ~ok_l, ~ok_f & ok_l],
                                ["Ambos aciertan", "Solo el ensamble acierta", "Solo LR acierta"], default="Ambos fallan")
print("Aciertos y errores de los dos modelos por clase real (Test):")
display(pd.crosstab(analisis["real"], analisis["acierto"], margins=True))

vars_perfil = ["Curricular units 1st sem (approved)", "Curricular units 2nd sem (approved)",
               "Curricular units 2nd sem (grade)", "Tuition fees up to date", "Debtor", "Scholarship holder",
               "Age at enrollment"]
grupos = analisis.groupby(["acierto", "real"], observed=True)
perfil = grupos[vars_perfil].mean()
perfil.insert(0, "n", grupos.size())
display(perfil.round(2))

**Resultados.**
- **Referencias (CV de 5 folds, F1 macro):** regresión logística 0.7170, SVM lineal 0.7086, polinomial 0.7211 y RBF 0.7217 (optimizados en la práctica anterior); el árbol de decisión único, 0.6559.
- **Ensambles con hiperparámetros base:** de 0.678 (AdaBoost) a 0.730 (Gradient Boosting); salvo AdaBoost, *bagging* y *pasting*, están en un rango de 0.016, menor que la desviación entre folds. El EBM sin interacciones (0.7143) es equivalente a la regresión logística; con 10 interacciones sube a 0.7285.
- **Optimizados (CV):** de 0.7101 (Bagging) a 0.7375 (*Voting soft*), una ganancia promedio de +0.013 frente a los base y con más sobreajuste en 9 de los 10 (gap de 0.099 a 0.163 en promedio). **Ensamble final: *Voting (soft)* con pesos (LR, SVM, RF, GB, HGB) = (1, 1, 2, 2, 2)**, con F1 macro de 0.7375 ± 0.0224, Accuracy 0.7858, MCC 0.6508 y ROC-AUC 0.9018.
- **CV repetida 5 × 5:** *Voting soft* supera a la regresión logística por +0.0160 de F1 macro (IC 95% [0.002, 0.030]; p = 0.024, p de Holm = 0.26). Ninguno de los 10 ensambles es significativamente mejor en F1 macro tras corregir; en Accuracy sí hay diferencias claras: *Voting soft* (+0.025) y el SVM RBF (+0.023), con p ≤ 0.002 sin corregir. El optimismo de las búsquedas es de 0.002 a 0.009.
- **Test (885 estudiantes):** *Voting soft* tiene Accuracy 0.7559, F1 macro 0.7028 [0.669, 0.734], MCC 0.6057 y ROC-AUC 0.8900, contra 0.7254, 0.6832, 0.5704 y 0.8838 de la regresión logística con su regla. La diferencia es de +0.031 de Accuracy (IC [0.010, 0.051]; McNemar p = 0.004) y de +0.020 de F1 macro (IC [−0.005, 0.043]).
- **El mejor modelo en Test fue otro:** Random Forest (F1 0.7161), seguido de Bagging (0.7130) y Pasting (0.7096), que en CV ocupaban los lugares 7, 10 y 8 de 10. La correlación entre el orden en CV y en Test es de −0.32 (p = 0.37).
- ***Dropout* y *Enrolled*:** el ensamble final y la regresión logística con regla detectan los mismos 203 de 284 *Dropout* (diferencia de F1 en esa clase de −0.001, IC [−0.024, 0.020]). En *Enrolled*, el ensamble reconoce 53% (85 de 159), la regresión logística con regla 57%, el SVM RBF 40% y el EBM 67%.
- **Generalización:** el F1 baja 0.031 de CV a Test en promedio, lo mismo que en la regresión logística (0.029); el gap en entrenamiento (0.06 a 0.21 en los ensambles) no predice esa pérdida.
- **Variables importantes:** unidades aprobadas en ambos semestres y colegiatura al corriente, en todos los métodos (MDI, permutación y EBM).
- **Literatura:** con el mismo Test, los ensambles quedan dentro del ruido de los modelos de Praptomo et al. (F1 macro de 0.683 a 0.701); las cifras de Villar y de Andrade (0.81 a 0.86) no son comparables.

**Análisis de errores (Test).**
- **El ensamble final y la regresión logística con regla aciertan juntos en 614 estudiantes y fallan juntos en 188**; el ensamble acierta en 55 donde la regresión logística falla, y la regresión logística en 28 donde el ensamble falla.
- **Dónde gana cada uno:**
  - el ensamble gana sobre todo en *Graduate*: 38 *Graduate* que solo él acierta, contra 5 de la regresión logística. Son titulados con un rendimiento algo menor que el típico (6.26 y 5.84 unidades aprobadas, contra 6.59 y 6.57 de los *Graduate* que ambos aciertan) que la regresión logística ponderada manda a *Enrolled*;
  - la regresión logística gana en *Enrolled*: 16 que solo ella acierta, contra 10 del ensamble;
  - en *Dropout* empatan: 7 que solo acierta el ensamble y 7 que solo acierta la regresión logística.
- **Dropouts que ninguno detecta (74):** aprueban en promedio 4.28 y 4.20 unidades en cada semestre (contra 1.83 y 1.07 en los 196 que ambos detectan), tienen un promedio de 11.48 en el 2º semestre (contra 3.85), todos tienen la colegiatura al corriente (contra 53%) y solo 8% son deudores (contra 28%). Se parecen a los *Enrolled* y *Graduate* en sus datos académicos y económicos: abandonan sin señales en los datos. Cambiar de modelo no los rescata: **el límite está en la información disponible, no en el algoritmo.**
- ***Enrolled* que ninguno acierta (58):** aprueban 4.50 y 4.07 unidades, en plena zona intermedia, y 29% son becarios (contra 7% de los *Enrolled* que ambos aciertan). **Los *Graduate* que ninguno acierta (56)** aprueban 4.98 y 4.45 unidades, más cerca de un *Enrolled* que de un titulado típico.

**Qué produjo este notebook.** Todas las tablas (`outputs/tablas/04_*`) y figuras (`outputs/figs/04_*`) de resultados. La evaluación en Test se realizó una sola vez,
sobre modelos y reglas ya fijados con Train.